# DiVaST: replication notebook

Reproduces **DiVaST-3L-768H**, the three-layer student of the paper, on Google Colab
(Runtime > Change runtime type > T4 GPU) or any machine with a CUDA GPU. Every step
calls the scripts in `src/` that produced the paper's results, with the same arguments:

1. data split (the validation indices are in `data/val_indices.json`)
2. teacher: CodeBERTa-small with tag-preserving truncation, or download it
3. student: keep teacher layers 2, 4, 6 and distill (alpha=0.5, T=2)
4. official evaluation of the challenge (batch 600, 10 repetitions)
5. comparison with the paper's numbers
6. classify new functions with the released model

On an RTX 4090 the student trains in under 20 minutes; a T4 is several times slower. Set `SMOKE_TEST = True` to check that every cell runs.

In [ ]:
# ---- What to run -----------------------------------------------------------------
TRAIN_TEACHER = False   # False: download the paper's teacher (NafiuRahman/divast, teacher/)
SMOKE_TEST = False      # True: tiny run that only checks that every cell works
EVALUATE_BASELINE = False  # True: also evaluate the organizers' CodeBERT baseline (+45 min on a T4)
RELEASED = "NafiuRahman/divast"

In [ ]:
import os, subprocess, sys
if not os.path.exists("src/train.py"):          # on Colab: get the code
    subprocess.run(["git", "clone", "https://github.com/nafiurahman00/divast"], check=True)
    os.chdir("divast")
# Install the Hugging Face libraries of the paper; keep the runtime's torch, numpy, pandas.
HF_LIBS = ("transformers", "datasets", "accelerate", "huggingface_hub")
pins = [l.strip() for l in open("requirements.txt") if l.startswith(HF_LIBS)]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=True)

SMOKE_TRAIN = ["--subset", "400", "--epochs", "1", "--max-len", "128"] if SMOKE_TEST else []
SMOKE_EVAL = ["--limit", "600", "--repetitions", "1"] if SMOKE_TEST else []

def sh(*args):
    print("$", " ".join(args), flush=True)
    subprocess.run(list(args), check=True)

## 1. Data split

`src/train.py` holds out the 5% of the training split listed in `data/val_indices.json`
(jointly stratified on both labels, seed 42) for checkpoint selection. The public test
split is used only for the final evaluation. `python src/prepare_data.py` rebuilds the
file.

In [ ]:
import json
idx = json.load(open("data/val_indices.json"))["indices"]
print(f"{len(idx):,} validation functions")

## 2. Teacher

CodeBERTa-small (6 layers, hidden size 768) whose tokenizer keeps the **end** of long
functions (`src/make_left_base.py`), fine-tuned with binary cross-entropy for 3 epochs,
learning rate 3e-5, batch size 32, inputs of 400 tokens, fp16, seed 42.

In [ ]:
if TRAIN_TEACHER:
    sh(sys.executable, "src/make_left_base.py")
    sh(sys.executable, "src/train.py", "--model", "models/_base_codeberta_left",
       "--output-dir", "models/teacher", "--loss", "bce", "--epochs", "3", "--lr", "3e-5",
       "--max-len", "400", "--bs", "32", "--fp16", "--save-steps", "2000", *SMOKE_TRAIN)
else:
    import shutil
    from huggingface_hub import snapshot_download
    path = snapshot_download(RELEASED, allow_patterns=["teacher/*"])
    shutil.copytree(os.path.join(path, "teacher"), "models/teacher", dirs_exist_ok=True)
print(sorted(os.listdir("models/teacher")))

## 3. Student (DiVaST-3L-768H)

`--student-layers 1,3,5` (0-based) keeps teacher layers 2, 4, and 6 together with the
embeddings and the classification head. The loss is
`(1 - alpha) * BCE(sigmoid(z_s), y) + alpha * T^2 * BCE(sigmoid(z_s / T), sigmoid(z_t / T))`
with alpha=0.5 and T=2; the teacher is frozen. The other settings are the teacher's.

In [ ]:
sh(sys.executable, "src/train.py", "--model", "models/teacher", "--output-dir", "models/kd_L3_135",
   "--loss", "bce", "--epochs", "3", "--lr", "3e-5", "--max-len", "400", "--bs", "32",
   "--student-layers", "1,3,5", "--teacher", "models/teacher",
   "--fp16", "--save-steps", "2000", *SMOKE_TRAIN)

## 4. Official evaluation

`src/evaluate_submission.py` re-implements the evaluation loop of the organizers'
`CodeBERT_baseline.ipynb`: inputs padded to 512 tokens, batch 600, fp16, 10 timed
repetitions with the PyTorch profiler, threshold 0.5, and the competition score.

In [ ]:
sh(sys.executable, "src/evaluate_submission.py", "--model", "models/kd_L3_135", "--batch-size", "600", *SMOKE_EVAL)
if EVALUATE_BASELINE:
    sh(sys.executable, "src/evaluate_submission.py", "--model",
       "NLBSE/nlbse27-code-comment-classification-baseline", "--batch-size", "600", *SMOKE_EVAL)

## 5. Comparison with the paper

The paper reports the run in `results/t4/` (one Colab T4 session for all models). F1
depends on the training run: seeds 42, 1, and 2 gave average F1 0.9285 to 0.9299 on an
RTX 4090 (`results/rtx4090/eval/`). Runtime depends on the GPU and the session.

In [ ]:
def row(path):
    d = json.load(open(path)); t = {r["cat"]: r["f1"] for r in d["per_task"]}
    return (f"F1 vuln {t['Vulnerability']:.4f} | F1 SATD {t['MAT']:.4f} | avg {d['mean_f1']:.4f} | "
            f"runtime {d['avg_runtime_s']:.2f}s | GFLOPs {d['avg_gflops']:,.0f} | score {d['score']:.4f}")
print("this run :", row("results/eval_kd_L3_135.json"))
print("paper    :", row("results/t4/eval_divast_3L_768H.json"))

## 6. Classify new functions

The released student loads like the challenge baseline. The labels are
`[Vulnerability, SATD]`, predicted when the sigmoid probability exceeds 0.5.

In [ ]:
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer

tok = RobertaTokenizer.from_pretrained(RELEASED)
model = RobertaForSequenceClassification.from_pretrained(RELEASED).eval()

def classify(functions):
    enc = tok(functions, return_tensors="pt", truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        p = torch.sigmoid(model(**enc).logits)
    return [{"vulnerability": bool(a > 0.5), "satd": bool(b > 0.5)} for a, b in p.tolist()]

classify(["int f(char *s) { /* TODO: check the length */ char b[8]; strcpy(b, s); return 0; }",
          "int add(int a, int b) { return a + b; }"])